In [ ]:
from langgraph.store.base import BaseStore
from langgraph.store.memory import InMemoryStore
from dotenv import load_dotenv
from langchain_core.runnables import RunnableConfig
from langchain_core.messages import SystemMessage,HumanMessage
from langgraph.graph import StateGraph ,START,END,MessagesState
from langchain_groq import ChatGroq
load_dotenv()

In [ ]:
model = ChatGroq(model="openai/gpt-oss-20b",
    temperature=0.7)

In [ ]:
store = InMemoryStore()

user_id = "user-1"

namespace = ("user",user_id,"details")

store.put(namespace , "user-detail-1",{"data":" Name - Saransh"})
store.put(namespace , "user-detail-2",{"data":"is pursing btech"})
store.put(namespace , "user-detail-3",{"data":"knowledge about langchain and langgraph ,django ,drf "})
store.put(namespace , "user-detail-4",{"data":"likes to code in python"})
store.put(namespace , "user-detail-5",{"data":"building chatbot and have been building rest api ."})
store.put(namespace , "user-detail-6",{"data":"is intersted in football"})

In [ ]:
SYS_PROMPT = """
You are an assistant with access to long-term memory about the user.

tailor your responses based on what you  know about the user.

Your goal is to provide relevant, friendly, and tailored
assistance that reflects the userls preferences, context, and past interactions.

If the user's name or relevant personal context is available, always personalize your responses by:
Always Address the user by name (e.g., "Sure, alex ... ") when appropriate
Referencing known projects, tools, or preferences .(e.g., "your MCP server python based project")
Adjusting the tone to feel friendly, natural, and directly aimed at the user

Avoid generic phrasing when personalization is possible. For example, instead of ."In TypeScript apps ... ".
say "Since your project is .built with TypeScript ... "

Use personalization especially in:
Greetings and transitions
Help or guidance tailored to tools and frameworks the user has worked with


integrate the user's details inside your answer if the detail is realatable.
Answer the user's question clearly and completely. At the end of every response, ask exactly three useful forward-looking questions that are relevant to the user's request and, when appropriate, connected to the user's remembered interests, goals, skills, or ongoing projects. Do not ask generic questions or repeat information already provided. here are the user's memories present (if available)-{users_memory_details}
"""

In [ ]:
def chat_node(state: MessagesState, config: RunnableConfig, store: BaseStore):
    configurable = config.get("configurable") or {}
    user_id = configurable.get("user_id", "user-1")
    namespace = ("user", user_id, "details")

    items = store.search(namespace)

    
    if items:
        users_memory_details = "\n".join(f"- {i.value.get('data', '')}" for i in items)
    else:
        users_memory_details = ""

    
    sys_prompt = SYS_PROMPT.format(users_memory_details=users_memory_details)
    sys_prompt = SystemMessage(sys_prompt)
    res = model.invoke([sys_prompt]+state['messages'])
    return {"messages":[res]}



In [ ]:
graph = StateGraph(MessagesState)

graph.add_node("chat_node",chat_node)
graph.add_edge(START,"chat_node")

final_graph = graph.compile(store=store)
final_graph

In [ ]:
config = {'configurable':{'user_id':"user-1"}}

result = final_graph.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "tell me about langchain in brief",
            }
        ]
    },
    config=config,
)

result['messages'][-1].content